# Ford VinGuard / Dock 360 - Pipeline Completo (Dados Reais)

Notebook consolidado para execucao ponta a ponta usando dados reais Ford Brasil.

In [1]:
import os
from pathlib import Path

# Fix CWD
notebook_dir = Path.cwd()
if notebook_dir.name == 'notebooks':
    os.chdir(notebook_dir.parent)
print(f"CWD: {os.getcwd()}")

import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import mlflow
from src.pipeline import feature_engineering_real, clustering_real, train_classifier_real, train_churn_real, visualizations

print("Imports OK")

CWD: d:\Abner\Downloads\FORD\Ford_Dock360


c:\Users\Adm\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK


## 1. Feature Engineering (Agregacao por VIN)

Transforma ordens de servico em um dataset de clientes (VINs).

In [2]:
feature_engineering_real.main()

Carregando data/raw/vin_share_Desafio_02.xlsx...
Carregado: 602,788 linhas, 25 colunas
Data de corte: 2024-10-31
Fim da janela futura (18m): 2026-04-30
Maior ServiceDate na base: 2026-05-04

=== VALIDACAO SNAPSHOT POS-VENDA ===
VINs com historico ate corte: 118,379

Distribuicao de churn_futuro_18m:
churn_futuro_18m
1    0.6459
0    0.3541
Name: proportion, dtype: float64

Distribuicao por modelo (top 10):
modelo
KA              50319
RANGER          39039
ECOSPORT        15987
TERRITORY        4120
BRONCO SPORT     2891
TRANSIT          2220
MAVERICK         1920
MUSTANG           873
F-150             610
CARGO             125
Name: count, dtype: int64

Missing values relevantes:
  km_max_ate_corte: 418 (0.35%)
  dias_ate_primeira_revisao: 0 (0.00%)
  intervalo_medio_revisoes_dias_ate_corte: 32,915 (27.80%)
  ano_modelo: 1 (0.00%)

Salvo em: data/processed/snapshots_pos_venda.csv
Salvo em: data/processed/dataset_churn_pos_venda.csv
Shape: (118379, 30)


,VIN_Hash,qtde_revisoes_ate_corte,primeiro_servico_ate_corte,ultimo_servico_ate_corte,modelo,ano_modelo,dealer_code_ate_corte,n_dealers_usados_ate_corte,km_max_ate_corte,sales_date,...,qtd_servicos_pos_corte,primeiro_servico_pos_corte,ultimo_servico_pos_corte,voltou_pos_corte,churn_futuro_18m,data_corte,fim_janela_churn,data_max_observada,janela_futura_observavel,churn
0,00008eef200a0a71fd52b4b0bd43c6e275b50a8fe56a6f...,1,2022-03-03,2022-03-03,KA,2021.0,3050,1,11010.0,2021-01-22,...,0,NaT,NaT,False,1,2024-10-31,2026-04-30,2026-05-04,True,1
1,00015a2ad3fd2a788fb2ccf83b443e5326b3d835b53468...,1,2020-11-16,2020-11-16,RANGER,2020.0,2606,1,9462.0,2020-02-10,...,0,NaT,NaT,False,1,2024-10-31,2026-04-30,2026-05-04,True,1
2,0002a2d7b0e64c34a5cd3fdc69c387d28128474230d897...,3,2022-04-13,2023-09-21,KA,2021.0,5650,1,50605.0,2020-11-09,...,0,NaT,NaT,False,1,2024-10-31,2026-04-30,2026-05-04,True,1
3,00031d6a4d99a0ccf6222ea9beb894e562ec5749603d92...,3,2022-01-10,2024-01-25,ECOSPORT,2021.0,6163,1,30251.0,2021-01-15,...,1,2025-01-31,2025-01-31,True,0,2024-10-31,2026-04-30,2026-05-04,True,0
4,00033ec3f308cce9068684af00cfa6072fed00f9567989...,3,2021-11-19,2022-10-17,KA,2021.0,6137,2,32473.0,2020-12-23,...,0,NaT,NaT,False,1,2024-10-31,2026-04-30,2026-05-04,True,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118374,fffd8585909c4b1220a7f51b9f22021001946a8716d7c0...,6,2023-06-20,2024-10-07,RANGER,2023.0,5292,1,60032.0,2023-01-20,...,5,2025-01-13,2026-02-23,True,0,2024-10-31,2026-04-30,2026-05-04,True,0
118375,fffe3818924544e4c0e31a7f66fd773ba6e6e090ad0708...,1,2024-03-18,2024-03-18,RANGER,2023.0,5671,1,13477.0,2023-04-29,...,1,2025-08-06,2025-08-06,True,0,2024-10-31,2026-04-30,2026-05-04,True,0
118376,fffe94a0374d392bcb1a2919641a6ce39c5edb4eaeaaf5...,2,2023-07-18,2024-02-08,TRANSIT,2022.0,1054,1,44324.0,2022-07-28,...,1,2026-02-04,2026-02-04,True,0,2024-10-31,2026-04-30,2026-05-04,True,0
118377,fffeac8612b7b562be46e48decbc8373c3d740be4cd6df...,3,2024-01-17,2024-01-17,RANGER,2023.0,3028,1,5948.0,2023-02-14,...,9,2025-01-27,2026-03-13,True,0,2024-10-31,2026-04-30,2026-05-04,True,0


## 2. Clustering (Segmentacao de Perfil)

Agrupa clientes por comportamento de manutencao.

In [3]:
clustering_real.run_clustering()

Carregado: 118,379 VINs

=== Selecao de k via Elbow + Silhouette ===
  k=2: inertia=624165, silhouette=0.2942
  k=3: inertia=524615, silhouette=0.2402
  k=4: inertia=439238, silhouette=0.2753
  k=5: inertia=373036, silhouette=0.3033
  k=6: inertia=334309, silhouette=0.3138
  k=7: inertia=302703, silhouette=0.3189
  k=8: inertia=282881, silhouette=0.3257
Salvo: reports/elbow_silhouette_pos_venda.png

=== Treinando segmentador K-Means pos-venda com k=4 ===

=== Centroides dos clusters (medias) ===
         qtde_revisoes_ate_corte  meses_desde_ultimo_servico_ate_corte  \
cluster                                                                  
0                           1.79                                 22.53   
1                           7.69                                  7.95   
2                           2.99                                 14.60   
3                           2.07                                 28.47   

         meses_relacionamento_ate_corte  n_dealers_usa

(                                                 VIN_Hash  cluster_raw  \
 0       00008eef200a0a71fd52b4b0bd43c6e275b50a8fe56a6f...            0   
 1       00015a2ad3fd2a788fb2ccf83b443e5326b3d835b53468...            0   
 2       0002a2d7b0e64c34a5cd3fdc69c387d28128474230d897...            2   
 3       00031d6a4d99a0ccf6222ea9beb894e562ec5749603d92...            2   
 4       00033ec3f308cce9068684af00cfa6072fed00f9567989...            0   
 ...                                                   ...          ...   
 118374  fffd8585909c4b1220a7f51b9f22021001946a8716d7c0...            1   
 118375  fffe3818924544e4c0e31a7f66fd773ba6e6e090ad0708...            0   
 118376  fffe94a0374d392bcb1a2919641a6ce39c5edb4eaeaaf5...            0   
 118377  fffeac8612b7b562be46e48decbc8373c3d740be4cd6df...            0   
 118378  ffff6da1a18ea062a9d373dbbcbbfd7a77435b64b0810c...            1   
 
        segmento_pos_venda  
 0       baixo_engajamento  
 1       baixo_engajamento  
 2         

## 3. Treinamento de Modelos (Churn e Perfil)

In [4]:
print("Treinando Churn...")
train_churn_real.train_churn_model()

print("\nTreinando Classificador de Perfil...")
train_classifier_real.train_all_models()

Treinando Churn...
Dataset: 118,379 VINs
Features pos-venda: ['ano_modelo', 'qtde_revisoes_ate_corte', 'meses_desde_ultimo_servico_ate_corte', 'meses_relacionamento_ate_corte', 'n_dealers_usados_ate_corte', 'km_max_ate_corte', 'pct_agenda_ate_corte', 'intervalo_medio_revisoes_dias_ate_corte', 'dias_ate_primeira_revisao', 'idade_veiculo_meses_ate_corte', 'modelo']
Distribuicao de churn_futuro_18m:
churn_futuro_18m
1    0.646
0    0.354
Name: proportion, dtype: float64

=== Treinando RandomForest com calibracao isotonica (cv=5) ===
Esse passo treina o modelo 5 vezes e pode demorar alguns minutos...

=== Avaliacao holdout aleatoria estratificada ===
Nota: para avaliacao final, prefira multiplos snapshots e split temporal.
AUC-ROC: 0.9446

              precision    recall  f1-score   support

    no_churn       0.83      0.82      0.82      8384
       churn       0.90      0.91      0.90     15292

    accuracy                           0.88     23676
   macro avg       0.86      0.86   

,model,f1_macro_cv_mean,f1_macro_cv_std,f1_macro_test
0,LogReg,0.413400,0.002161,0.412752
2,RandomForest,0.408610,0.002484,0.411011
1,DecisionTree,0.401867,0.019215,0.390045


## 4. Visualizacoes e Relatorios

In [5]:
visualizations.main()

Visualizacoes geradas em reports/
